# 1.3 金融市场基础

> **这一节讲什么？**  
> 写策略之前，你得先搞清楚：**你在什么市场里交易？买的是什么？怎么下单？交易需要付多少手续费？**
> 这些「市场规则」听起来无聊，但会直接影响你的策略能不能跑通。

## 学习目标
- 了解主要金融资产类别：股票、ETF、期货、期权
- 理解 K 线图的读法
- 熟悉订单类型：市价单 vs 限价单
- 理解交易成本对策略收益的侵蚀

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['figure.figsize'] = (12, 5)

## 1. 主要资产类别

金融市场里有各种「标的」可以交易。对初学者来说，最常接触的是股票和 ETF；更进阶的是期货和期权。

**先知道有哪些选择，以及它们的核心差别：**

| 资产 | 通俗理解 | 特点 | 示例 |
|------|---------|------|------|
| **股票 (Stock)** | 买了公司的一小块所有权 | 高风险高收益，日内可多次交易 | AAPL（苹果）, 贵州茅台 |
| **ETF** | 一篮子股票打包成一只基金 | 分散化、低费率，比买单只股票风险低 | SPY（标普500）, 510050 |
| **期货 (Futures)** | 约定以某价格在未来某天买/卖东西 | 有杠杆（风险倍增）、有到期日、可做空 | ES（标普500期货）, IF |
| **期权 (Options)** | 花少量钱买一个「选择权」，但不是义务 | 收益不对称（最多亏权利金），灵活对冲 | SPY Put（看跌期权）|
| **债券 (Bonds)** | 借给政府/公司钱，收固定利息 | 风险低，收益稳定，与股票相关性低 | 国债, TLT |
| **外汇 (FX)** | 两种货币之间互换 | 24小时交易、流动性极高 | EUR/USD（欧元兑美元）|
| **加密货币** | 去中心化数字资产 | 极高波动率、永不休市（7×24h）| BTC, ETH |

> **初学者建议：** 先从股票和 ETF 开始，规则最清晰、监管最完善。期货和期权有杠杆，搞懂之前不要碰。

## 2. 股票 K 线图解读

### 为什么要看 K 线而不是普通折线图？

普通折线图只有收盘价一个信息。K 线图能在同一根「蜡烛」里展示**四个价格**：

```
        |   ← 上影线：最高价到「实体顶部」的距离
      ┌─┴─┐
      │   │  ← 实体（绿色=涨，红色=跌）
      │   │     实体顶 = max(开盘, 收盘)
      └─┬─┘     实体底 = min(开盘, 收盘)
        |   ← 下影线：实体底部到最低价的距离
```

**如何解读一根 K 线：**
- **实体大** → 当天有强烈的买/卖力量
- **长上影线** → 盘中涨上去了但被打下来（空方压制）
- **长下影线** → 盘中跌下去了但被推回来（多方支撑）
- **十字星（实体极小）** → 多空力量均衡，可能是趋势转折点

In [ ]:
import yfinance as yf

# 下载近30个交易日数据
df = yf.download('AAPL', period='60d', progress=False)
df = df.tail(30).copy()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), 
                                gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

# K线图
for idx, (date, row) in enumerate(df.iterrows()):
    o, h, l, c = float(row['Open']), float(row['High']), float(row['Low']), float(row['Close'])
    color = 'green' if c >= o else 'red'
    # 实体（开盘到收盘的矩形）
    rect = mpatches.FancyBboxPatch((idx - 0.3, min(o, c)), 0.6, abs(c - o),
                                    color=color, alpha=0.8)
    ax1.add_patch(rect)
    # 上下影线
    ax1.plot([idx, idx], [l, min(o, c)], color=color, linewidth=1)
    ax1.plot([idx, idx], [max(o, c), h], color=color, linewidth=1)

ax1.set_xlim(-1, len(df))
ax1.set_ylim(df['Low'].min().item() * 0.99, df['High'].max().item() * 1.01)
ax1.set_title('AAPL K线图（近30交易日）— 绿色=涨（收>开），红色=跌（收<开）', fontsize=13)
ax1.set_ylabel('价格 (USD)')
ax1.grid(alpha=0.2)

# 成交量（颜色对应涨跌）
for idx, (date, row) in enumerate(df.iterrows()):
    o, c = float(row['Open']), float(row['Close'])
    color = 'green' if c >= o else 'red'
    ax2.bar(idx, float(row['Volume']) / 1e6, color=color, alpha=0.7, width=0.8)

ax2.set_ylabel('成交量 (百万股)')
ax2.set_title('成交量 — 放量上涨更可信；缩量反弹要谨慎', fontsize=11)
ax2.grid(alpha=0.2)

# X轴日期标签
step = max(1, len(df) // 8)
ax2.set_xticks(range(0, len(df), step))
ax2.set_xticklabels([df.index[i].strftime('%m-%d') for i in range(0, len(df), step)],
                     rotation=30)
plt.tight_layout()
plt.show()

## 3. 订单类型

### 为什么下单方式很重要？

想象你在农贸市场买苹果：
- **市价单（Market Order）**：「老板，给我来一斤，你说多少钱就多少钱！」——保证买到，但价格未知（在流动性差的市场里可能多付很多）
- **限价单（Limit Order）**：「老板，苹果 3.5 元/斤我就买，贵了我不买」——价格有保障，但可能买不到（价格没跌到那里）
- **止损单（Stop Order）**：「如果苹果跌到 3 元以下，我就赶快买！」（或做空时：「如果涨到 10 元以上，我就赶快割肉！」）

| 订单类型 | 通俗理解 | 优点 | 缺点 |
|----------|---------|------|------|
| **市价单 (Market Order)** | 「立刻成交，价格随便」 | 保证成交，速度最快 | 价格不可控，高波动时滑点大 |
| **限价单 (Limit Order)** | 「这个价格成交，否则不买」 | 控制价格，不会多付 | 可能等不到，错过交易机会 |
| **止损单 (Stop Order)** | 「跌破某价格就触发，马上卖」 | 自动保护，控制最大损失 | 触发后变市价单，极端行情滑点大 |
| **止损限价单** | 止损触发后变限价单 | 结合两者 | 极端行情可能根本无法成交 |

> **量化交易的实践：** 回测时通常用收盘价成交（理想化），实盘时要考虑滑点（Slippage），实际成交价会比预想的差一点。

## 4. 交易成本的影响

### 为什么策略在回测好，实盘就差？

这是量化新手最常见的坑：忘记算交易成本。

每次买卖，你都要付：
- **手续费（Commission）**：券商收的佣金，美股约 0.03%~0.1%/笔，A 股约 0.03%~0.3%
- **滑点（Slippage）**：你想按 100 元买，实际成交了 100.05 元（买卖价差）
- **冲击成本（Market Impact）**：大单买入会把价格推高

**一笔 0.2% 的成本，交易 200 次就是 40% 的成本。** 收益率本来 50%，扣完成本就剩 10% 了……

下面的模拟让你直观看到：同样一个有盈利能力的策略（55% 胜率），不同手续费吃掉多少利润：

In [ ]:
# 模拟交易成本对策略的影响
np.random.seed(42)
n_trades = 200
# 假设每笔交易有 55% 的胜率，盈亏比 1:1
win_rate = 0.55
win_amount = 0.01   # 每次盈利 1%
loss_amount = -0.01  # 每次亏损 1%

outcomes = np.random.choice([win_amount, loss_amount], 
                              size=n_trades, p=[win_rate, 1 - win_rate])

# 不同手续费率
commission_rates = [0, 0.001, 0.002, 0.005]
labels = ['无成本（理想回测）', '0.1%（低佣金）', '0.2%（常见水平）', '0.5%（偏高，会亏钱）']

fig, ax = plt.subplots(figsize=(12, 5))
for rate, label in zip(commission_rates, labels):
    net = outcomes - rate  # 每笔扣除手续费
    cum = (1 + net).cumprod() * 10000
    ax.plot(cum, label=label, linewidth=1.5)

ax.axhline(10000, color='gray', linestyle='--', alpha=0.5)
ax.set_title(f'200笔交易 × 胜率{win_rate:.0%} × 盈亏比1:1：交易成本的侵蚀', fontsize=13)
ax.set_xlabel('交易次数')
ax.set_ylabel('资产价值')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print('💡 结论：高频策略对手续费极其敏感；频率越高，成本的影响越大！')
print('💡 量化回测时必须加入真实的手续费假设，否则回测结果毫无参考价值。')

## 🎯 练习

1. 观察 K 线图中的**长上影线**和**长下影线**分别意味着什么样的多空博弈？
2. 将上方模拟中 `win_rate` 改为 0.50（完全随机），观察不同手续费下的结果。在没有信息优势的情况下，手续费本身就决定了你一定会亏钱。
3. **思考**：如果一个策略每天交易 10 次，手续费 0.1%，需要多高的日胜率才能覆盖成本？（提示：每天成本 = 10 × 0.2% = 2%，每笔需要净盈利 0.2% 才能回本）

---
**下一模块** → `../02_data/01_data_sources.ipynb`